In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [3]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Nice! Leftover chicken and rice open up several quick, tasty options. Here are recipe ideas I found that work well with those ingredients. Tell me which one you’d like full, step-by-step instructions for, and I’ll pull them up.

1) Leftover Chicken and Egg Fried Rice
- Why it’s good: Uses both leftovers, can be done in one pan in about 20 minutes; eggs add protein and texture; you can toss in frozen peas/corn and a splash of soy sauce.
- Typical setup: leftover chicken, cooked rice, eggs, peas, corn, red pepper, spring onions, garlic, soy sauce, sesame oil.
- Source: Easy Peasy Foodie
- Link: https://www.easypeasyfoodie.com/leftover-chicken-egg-fried-rice

2) Rotisserie Chicken and Rice (One-Pan)
- Why it’s good: Very quick weeknight option; uses rotisserie chicken you already have; one pan, minimal cleanup.
- Typical setup: onions, tomatoes, Cajun seasoning (or your preferred spice), chicken, cooked rice, chicken broth; finish in one skillet.
- Source: Laura Fuentes
- Link: https://ww

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='94bfd107-b972-4720-ac07-d1162870190b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 703, 'prompt_tokens': 199, 'total_tokens': 902, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EGra2P0fwoOfnHtRUjCz8vxJQbLHF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03a80-d736-7ac3-9ad3-dec560220487-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'leftover chicken and rice recipes'}, 'id': 'call_r5hOovuwxx1uCQtEVkfrenhz', 't

## Image Input

In [7]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [8]:
print(uploader.value)

({'name': 'pantry-food-image.png', 'type': 'image/png', 'size': 1769885, 'content': <memory at 0x1142213c0>, 'last_modified': datetime.datetime(2026, 8, 25, 20, 8, 31, 6000, tzinfo=datetime.timezone.utc)},)


In [9]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [12]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Give me recipe suggestions using ingredients in this image."},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

Nice assortment to work with. Here are recipe ideas that use items I can see in the cabinet image. Pick one and I’ll give you full, step-by-step instructions.

1) Chickpea, Corn, and Tomato Salad with Greek Dressing
- What to use from the image: Bush’s garbanzo beans, Libby’s/Del Monte corn (sweet corn), Del Monte diced tomatoes, Kraft Greek dressing.
- How it comes together: Drain and rinse the beans; drain the corn; combine with the tomatoes. Toss with Greek dressing and a pinch of salt/pepper. Serve chilled or at room temp. Add chopped onions or herbs if you have them.

2) Creamy Tomato-Chickpea Soup (quick pantry soup)
- What to use: Campbell’s Tomato Soup, garbanzo beans, chicken broth (or water), optional corn for sweetness.
- How it comes together: Sauté a bit if you have onions, then simmer the tomato soup with broth and beans until hot. Stir in corn at the end. Add pepper or paprika to taste.

3) Creamy Chicken and Corn Bake (one-pan casserole-style)
- What to use: Campbell’s 

## Audio Input

In [13]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.52it/s]


Done.


In [14]:
agent = create_agent(
    model='gpt-audio',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

I currently don't have the capability to view images. However, I can help you with an audio file. Please go ahead and upload the audio file, and I can assist by analyzing it for content, transcription, or other informational details. Feel free to share what specifically you’d like to learn from the audio.
